## Fine-tune CLTL/MedRoBERTa.nl for NAME + ADDRESS (hybrid pipeline)

Trains on `output/train.jsonl` (5022 documents), produced by the `azorg_skeleton.py` /
`pii_table.py` / `gen_templates.py` / `run_pipeline.py` synthetic pipeline. The dataset
carries 8 span labels (`NAME, AGE, DATE, INSZ, RIZIV, ADDRESS, PHONE, URL`), but **this
notebook only trains the model on NAME and ADDRESS** — the two labels whose surface forms
are genuinely unbounded and need learned context (`patterns.py`'s own design split, see its
module docstring: "MODEL-OWNED... unbounded surface forms, context is the only
discriminator").

The other six labels are format-constrained enough that a checksummed/validated regex beats
a fine-tuned transformer on both precision and recall, at zero training cost — `patterns.py`
detects them directly at inference time, not via this model:
- **AGE, DATE, INSZ, RIZIV, PHONE, URL** — regex-detected (`patterns.py::detect`)
- **GENDER** — never a text span at all; derived from INSZ digits 7-9 parity
  (`patterns.py::derive_gender`), odd=male even=female, only after mod-97 checksum
  validation

Spans for those six labels are still present in `output/train.jsonl` (so you can train an
all-in-one comparison model later if you ever want one), but are **filtered out at
label-set-construction time** (see "Build the BIO label set" below) — the NER model this
notebook actually trains only ever predicts `B-NAME`/`I-NAME`/`B-ADDRESS`/`I-ADDRESS`/`O`.

**Runs on CPU or GPU (e.g. Colab) unchanged** — batch size and fp16 are picked automatically
based on whether a GPU is available.

### Files needed on Colab

| File | Used by | Notes |
|---|---|---|
| `output/train.jsonl` | Load dataset cell | Preserve the `output/` subfolder, or edit `data_files` below to wherever you place it |
| `patterns.py` | Hybrid regex+NER eval cells | Plain stdlib-only Python module (no pip install needed) — must be importable |
| `pdf_audit.py` | org_data extraction cell | Plain Python module (needs `pymupdf`, auto-installed below) — must be importable |
| `org_data/*.pdf` (5 files) | Generalization check + hybrid eval cells | Keep the `org_data/` folder name, or edit `ORG_DATA = Path("org_data")` below |

`patterns.py`/`pdf_audit.py` can either be uploaded flat into `/content/` (Colab's plain
working directory), or kept on Google Drive — the setup cell below mounts Drive and adds
`DRIVE_MODULE_DIR` (default `/content/drive/MyDrive/TamerBERT`, edit if yours differs) to
`sys.path` so `import patterns` / `import pdf_audit` resolve either way. Everything else
(`transformers`, `datasets`, `evaluate`, `seqeval`, `accelerate`, `pymupdf`) is pip-installed
automatically by the setup cells when `ON_COLAB` is detected.

Set Runtime > Change runtime type > GPU, get the four files above in place, then run all cells
in order through the "Full training run" cell (and the evaluation cells after it, which
now require the model to have been trained and saved first).

**PDF extraction note**: org_data text is extracted via `pdf_audit.py` (PyMuPDF-based) rather
than `pdfplumber`. `pdf_audit.py`'s own diagnostics on these exact 5 PDFs showed pdfplumber
produces severe glyph-spacing damage on `HealthOne NOVA3.pdf` (52.6% single-character tokens, a
corrupted date), while PyMuPDF's text layer is clean on all 5. If your local `fitz` import
fails with `ModuleNotFoundError: No module named 'frontend'`, an unrelated PyPI package also
named `fitz` (a neuroimaging tool) is shadowing PyMuPDF — `pip uninstall fitz && pip install
pymupdf` fixes it. This shouldn't occur on a fresh Colab runtime.

## Setup (Colab-aware)

In [ ]:
import torch

try:
    ON_COLAB = "google.colab" in str(get_ipython())
except NameError:
    ON_COLAB = False

if ON_COLAB:
    %pip install -q transformers datasets evaluate seqeval accelerate
    # Upload the labeled dataset if it isn't already in the working directory:
    # from google.colab import files
    # files.upload()  # select output/train.jsonl (or adjust the data_files path below)

    # patterns.py / pdf_audit.py live on Drive rather than the plain /content/ working
    # directory -- mount it (no-op if already mounted) and add that folder to sys.path
    # so later `import patterns` / `import pdf_audit` resolve. Adjust the path below if
    # your files are stored elsewhere.
    import sys
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MODULE_DIR = "/content/drive/MyDrive/TamerBERT"
    if DRIVE_MODULE_DIR not in sys.path:
        sys.path.insert(0, DRIVE_MODULE_DIR)

HAS_GPU = torch.cuda.is_available()
USE_FP16 = HAS_GPU
TRAIN_BATCH_SIZE = 32 if HAS_GPU else 8
EVAL_BATCH_SIZE = 32 if HAS_GPU else 8

print("On Colab:", ON_COLAB)
print("GPU available:", HAS_GPU, "-", torch.cuda.get_device_name(0) if HAS_GPU else "none")
print("Batch size:", TRAIN_BATCH_SIZE, "| fp16:", USE_FP16)

## Load the labeled dataset and split train/validation

In [ ]:
from datasets import load_dataset

raw = load_dataset("json", data_files="output/train.jsonl")["train"]
split = raw.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]
len(train_ds), len(eval_ds)

## Build the BIO label set from the actual data

In [ ]:
# The dataset carries 8 span labels, but this model is only trained on NAME/ADDRESS --
# AGE/DATE/INSZ/RIZIV/PHONE/URL are regex-detected instead (see the hybrid layer below).
all_labels_in_data = sorted({e["label"] for row in raw for e in row["spans"]})
MODEL_LABELS = ["ADDRESS", "NAME"]
print("labels present in the data:", all_labels_in_data)
print("labels this model is trained on:", MODEL_LABELS)

tags = ["O"] + [f"{prefix}-{lbl}" for lbl in MODEL_LABELS for prefix in ("B", "I")]
label2id = {tag: i for i, tag in enumerate(tags)}
id2label = {i: tag for tag, i in label2id.items()}
print(len(label2id), "BIO tags")
label2id

## Tokenize and align labels

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "CLTL/MedRoBERTa.nl"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode_example(example):
    # Only NAME/ADDRESS spans become training targets -- AGE/DATE/INSZ/RIZIV/PHONE/URL
    # spans in the data are deliberately ignored here (tagged O), since patterns.py
    # handles those at inference time instead of the model.
    spans = [s for s in example["spans"] if s["label"] in MODEL_LABELS]

    enc = tokenizer(example["text"], truncation=True,
                    max_length=512, return_offsets_mapping=True)
    labels = []
    for (start, end) in enc["offset_mapping"]:
        if start == end:                # special tokens
            labels.append(-100)
            continue
        tag = "O"
        for s in spans:
            if start >= s["start"] and end <= s["end"]:
                prefix = "B-" if start == s["start"] else "I-"
                tag = prefix + s["label"]
                break
        labels.append(label2id[tag])
    enc["labels"] = labels
    enc.pop("offset_mapping")
    return enc

encoded_train = train_ds.map(encode_example, remove_columns=train_ds.column_names)
encoded_eval = eval_ds.map(encode_example, remove_columns=eval_ds.column_names)

## Model, metrics, and Trainer setup

In [ ]:
import numpy as np
import evaluate
from transformers import AutoModelForTokenClassification, DataCollatorForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label2id), id2label=id2label, label2id=label2id
)
data_collator = DataCollatorForTokenClassification(tokenizer)
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_predictions = [
        [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(preds, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(preds, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## Quick pipeline sanity check (a handful of steps, not a real training run)

Run this first to confirm everything wires up correctly before committing to the full run.
On CPU this takes ~1 minute; on GPU it's near-instant.

In [ ]:
from transformers import TrainingArguments, Trainer

smoke_args = TrainingArguments(
    output_dir="./ner_smoke_test",
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    fp16=USE_FP16,
    max_steps=5,
    logging_steps=1,
    report_to=[],
)

smoke_trainer = Trainer(
    model=model,
    args=smoke_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
smoke_trainer.train()

## Full training run

Once the smoke test above completes cleanly, run the real training here. Batch size and fp16
are already set from the GPU check above — 32 examples/step on GPU vs. 8 on CPU. Adjust
`num_train_epochs` if you want more/less training given how much faster GPU makes each step.

In [ ]:
training_args = TrainingArguments(
    output_dir="./medroberta-nl-pii-ner",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    fp16=USE_FP16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./medroberta-nl-pii-ner/final")
tokenizer.save_pretrained("./medroberta-nl-pii-ner/final")

## Out-of-template generalization check: run on org_data

`org_data`'s 5 letters (Fysische Geneeskunde, Oftalmologie, Pneumologie, Gastro-enterologie)
use a different structure than `synthetic_reports`, so this tests whether the model learned
genuine NAME/ADDRESS extraction or just memorized the training template. The model was only
trained on NAME/ADDRESS, so `entities` below will only ever contain those two labels — the
other six (AGE/DATE/INSZ/RIZIV/PHONE/URL) plus derived GENDER are covered in the hybrid layer
further down, which adds `patterns.py`'s regex detections back in for the full picture. Upload
the `org_data` folder (5 PDFs) to the Colab working directory before running this cell.

In [ ]:
if ON_COLAB:
    %pip install -q pdfplumber pymupdf
    # Upload org_data as a zip and unzip, or upload the 5 PDFs individually:
    # from google.colab import files
    # files.upload()
    # !mkdir -p org_data && mv *.pdf org_data/

import json
from pathlib import Path

import pdfplumber
import pdf_audit  # pdf_audit.py -- repair_extraction() only, see extract_text()'s docstring
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

EVAL_MODEL_PATH = "./medroberta-nl-pii-ner/final"
ORG_DATA = Path("org_data")

eval_tokenizer = AutoTokenizer.from_pretrained(EVAL_MODEL_PATH)
eval_model = AutoModelForTokenClassification.from_pretrained(EVAL_MODEL_PATH)
ner = pipeline("token-classification", model=eval_model, tokenizer=eval_tokenizer, aggregation_strategy="simple")

def extract_text(pdf_path):
    """pdfplumber for extraction, pdf_audit for cleanup -- NOT pdf_audit.extract().

    Verified empirically: org_data's redaction overlays (PATIENT_NAME/RESPONSIBLE_NAME/
    DOCTOR_NAME placeholders, the X-masked INSZ/RIZIV runs) are rendered as PDF objects
    that PyMuPDF's block iteration (both pdf_audit.extract()'s dict walk and even plain
    page.get_text("text")) returns out of visual reading order or drops from the header
    line entirely -- pdfplumber is the one extractor that places them correctly for this
    layout. PyMuPDF still wins on raw glyph-spacing damage (NOVA3 was 52.6% single-char
    tokens under plain pdfplumber), so repair_extraction() (letterspacing/page-furniture
    repair, pure text functions, extractor-agnostic) is layered on top instead of
    switching extractors outright -- best of both rather than an either/or choice.
    """
    with pdfplumber.open(pdf_path) as pdf:
        raw = "\n".join(page.extract_text() or "" for page in pdf.pages)
    repaired, _ = pdf_audit.repair_extraction(raw)
    return repaired

def run_chunked(text, max_tokens=400, stride=50):
    """RoBERTa caps at 512 tokens; chunk with overlap so nothing near a boundary is missed."""
    offsets = eval_tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)["offset_mapping"]
    entities, seen = [], set()
    start_tok = 0
    while start_tok < len(offsets):
        end_tok = min(start_tok + max_tokens, len(offsets))
        char_start, char_end = offsets[start_tok][0], offsets[end_tok - 1][1]
        for ent in ner(text[char_start:char_end]):
            abs_start, abs_end = char_start + ent["start"], char_start + ent["end"]
            key = (abs_start, abs_end, ent["entity_group"])
            if key in seen:
                continue
            seen.add(key)
            entities.append({
                "label": ent["entity_group"], "value": text[abs_start:abs_end],
                "start": abs_start, "end": abs_end, "score": round(float(ent["score"]), 4),
            })
        if end_tok == len(offsets):
            break
        start_tok = end_tok - stride
    return sorted(entities, key=lambda e: e["start"])

org_data_results = {}
for pdf_path in sorted(ORG_DATA.glob("*.pdf")):
    text = extract_text(pdf_path)
    entities = run_chunked(text)
    org_data_results[pdf_path.name] = entities
    print(f"\n=== {pdf_path.name} ({len(entities)} entities) ===")
    for e in entities:
        print(f"  {e['label']:16s} {e['value']!r:50s} score={e['score']}")

Path("org_data_model_predictions.json").write_text(
    json.dumps(org_data_results, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("\nSaved -> org_data_model_predictions.json")

## Hybrid regex + MedRoBERTa layer, with precision/recall/F1 on org_data

Final architecture: MedRoBERTa is fine-tuned on **NAME and ADDRESS only** (unbounded surface
forms, context is the only discriminator). Everything else is deterministic:

- **AGE, DATE, INSZ, RIZIV, PHONE, URL** — format-constrained enough that `patterns.py`'s
  checksummed/validated regex beats a transformer on both precision and recall, at zero
  training cost. Regex wins these categories outright; the model was never even trained on
  them (see "Build the BIO label set" above), so it structurally cannot predict them.
- **GENDER** — never a text span. Derived from INSZ digits 7-9 parity
  (`patterns.derive_gender`) after mod-97 checksum validation. Reported per document below,
  not scored via P/R/F1 like the span labels — there's no "GENDER span" to compare against.

Evaluation is **exact span match** (same start/end char offsets and label) per document, then
aggregated across all 5 `org_data` PDFs, both overall and per-label. Since ADDRESS is now
trained directly as one label (matching org_data's single opaque `[ADRES]` placeholder), no
STREET/ZIPCODE/CITY merging workaround is needed anymore — every label has real, non-vacuous
ground truth from this test set.

In [ ]:
import re
import patterns  # patterns.py -- checksummed/format-locked regex layer, see its docstring

# The model was trained on NAME/ADDRESS only (see "Build the BIO label set" above), so these
# six are regex-only in practice, not just by preference -- the model structurally cannot
# predict them.
REGEX_ONLY_LABELS = {"AGE", "DATE", "INSZ", "RIZIV", "PHONE", "URL"}
# GENDER is never a span label at all (derived from patterns.derive_gender, risk-context only)


def regex_detect(text):
    """patterns.py's real-format detector (checksummed INSZ/RIZIV, validated phone/date/age
    surfaces). Only keeps hits for the six regex-owned labels; DOB folds into DATE (patterns.py
    keeps it as a separate internal label for its own precedence/consumption logic, but it's
    the same DATE label everywhere else -- training data, this evaluation)."""
    hits = patterns.detect(text)
    entities = []
    for h in hits:
        label = "DATE" if h.label == "DOB" else h.label
        if label not in REGEX_ONLY_LABELS:
            continue
        if text[h.start:h.end] != h.text:
            continue  # confusable-folding shifted offsets; skip rather than mislabel
        entities.append({"label": label, "start": h.start, "end": h.end, "value": h.text})
    entities.sort(key=lambda e: e["start"])
    return entities


def merge_predictions(ner_entities, regex_entities):
    """NAME/ADDRESS (model) and the six regex-owned labels are completely disjoint label
    sets now -- the model was never trained on the regex labels, so there's nothing to
    reconcile beyond a defensive filter. Simple concatenation."""
    kept_ner = [e for e in ner_entities if e["label"] not in REGEX_ONLY_LABELS]
    merged = kept_ner + [e for e in regex_entities if e["label"] in REGEX_ONLY_LABELS]
    merged.sort(key=lambda e: e["start"])
    return merged


def report_gender(text):
    """GENDER is never a predicted span -- derived per document from INSZ parity, exactly as
    designed (patterns.py's DERIVED note). Returned separately from the span entities."""
    return patterns.derive_gender(text)


# --- org_data-specific supplement: recognizing ALREADY-REDACTED placeholders ---
# org_data's PDFs were masked by a previous process (XXXXXXXXXXX X-runs, [TELEFOON]/[URL]
# brackets) -- patterns.py correctly does NOT match these (no digits to checksum-validate),
# since it's built to find real PII, not recognize prior redaction. This supplement handles
# that one test set's specific style; it is NOT part of the general regex layer above.
MASKED_ID_RE = re.compile(r"X{6,}")
BRACKET_PHONE_RE = re.compile(r"\[TELEFOON\]")
BRACKET_URL_RE = re.compile(r"\[URL\]")


def detect_redacted_placeholders(text):
    entities = []
    masked_matches = list(MASKED_ID_RE.finditer(text))
    for i, m in enumerate(masked_matches):
        label = "INSZ" if i % 2 == 0 else "RIZIV"
        entities.append({"label": label, "start": m.start(), "end": m.end(), "value": m.group(0)})
    for m in BRACKET_PHONE_RE.finditer(text):
        entities.append({"label": "PHONE", "start": m.start(), "end": m.end(), "value": m.group(0)})
    for m in BRACKET_URL_RE.finditer(text):
        entities.append({"label": "URL", "start": m.start(), "end": m.end(), "value": m.group(0)})
    entities.sort(key=lambda e: e["start"])
    return entities

In [ ]:
PATIENT_PSEUDO_RE = re.compile(r"PATI[ËE]NT\s+([A-Z])\b")
RESP_PSEUDO_RE = re.compile(r"Verantwoordelijke\s+([A-Z]+)\b")
DOCTOR_PSEUDO_RE = re.compile(r"dr\.\s*ARTS_([A-Z]+)\b")
# Span includes the suffix ("50-jarige" / "78 jaar"), matching patterns.py's own tested AGE
# convention -- not just the bare digits (same fix applied to fill_org_data_placeholders.py).
AGE_RE1 = re.compile(r"\d{1,3}-jarige")
AGE_RE2 = re.compile(r"LEEFTIJD:\s*(\d{1,3}\s*jaar)")
# ORGANIZATION folds into NAME now -- a hospital/org name is PII the same way a person's is.
ORGANIZATION_RE = re.compile(r"\bAZORG\b")
# Boundary-guarded: at least one org_data PDF has a stray mid-word "p[ADRES]nd" artifact
# from the original redaction tool (it appears to have blanket-replaced the substring
# "adres" wherever found, including inside unrelated Dutch words). The negative lookaround
# requires whitespace/string-boundary on both sides, so only the real footer placeholder
# ("[ADRES] T [TELEFOON]") counts as gold, not that artifact.
ADDRESS_RE = re.compile(r"(?<!\S)\[ADRES\](?!\S)")


def merge_claims(claimed, candidates):
    for start, end, label, value in sorted(candidates, key=lambda c: (c[0], -(c[1] - c[0]))):
        if any(start < e and end > s for s, e, _, _ in claimed):
            continue
        claimed.append((start, end, label, value))


def ground_truth_org_data(text):
    claimed = []
    candidates = []
    # NAME is merged across patient/responsible/doctor/organization roles, matching flat9.
    candidates += [(m.start(), m.end(), "NAME", m.group(0)) for m in PATIENT_PSEUDO_RE.finditer(text)]
    candidates += [(m.start(), m.end(), "NAME", m.group(0)) for m in RESP_PSEUDO_RE.finditer(text)]
    candidates += [(m.start(), m.end(), "NAME", m.group(0)) for m in DOCTOR_PSEUDO_RE.finditer(text)]
    candidates += [(m.start(), m.end(), "NAME", m.group(0)) for m in ORGANIZATION_RE.finditer(text)]
    candidates += [(m.start(), m.end(), "AGE", m.group()) for m in AGE_RE1.finditer(text)]
    candidates += [(m.start(1), m.end(1), "AGE", m.group(1)) for m in AGE_RE2.finditer(text)]
    candidates += [(m.start(), m.end(), "ADDRESS", m.group(0)) for m in ADDRESS_RE.finditer(text)]
    # patterns.py's real-format detector (won't fire on org_data's masked style, but covers
    # the rare case a real value slipped through) plus the redacted-placeholder supplement
    # (X-runs, [TELEFOON]/[URL] brackets) which IS what org_data actually contains.
    candidates += [(e["start"], e["end"], e["label"], e["value"]) for e in regex_detect(text)]
    candidates += [(e["start"], e["end"], e["label"], e["value"]) for e in detect_redacted_placeholders(text)]

    merge_claims(claimed, candidates)
    claimed.sort(key=lambda c: c[0])
    return [{"label": l, "start": s, "end": e, "value": v} for s, e, l, v in claimed]

In [ ]:
def span_prf_counts(gold, pred):
    gold_set = {(e["start"], e["end"], e["label"]) for e in gold}
    pred_set = {(e["start"], e["end"], e["label"]) for e in pred}
    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)
    return tp, fp, fn


def prf(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) else 1.0
    recall = tp / (tp + fn) if (tp + fn) else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return precision, recall, f1


totals_ner = [0, 0, 0]
totals_merged = [0, 0, 0]
per_label_merged = {}
gender_report = {}

for pdf_path in sorted(ORG_DATA.glob("*.pdf")):
    text = extract_text(pdf_path)
    gold = ground_truth_org_data(text)
    ner_only = run_chunked(text)
    # org_data is already redacted (X-runs, [TELEFOON]/[URL]) -- patterns.py's real-format
    # detector won't fire on that, so include the redacted-placeholder supplement too,
    # matching what ground_truth_org_data itself does, for an apples-to-apples comparison.
    reg = regex_detect(text) + detect_redacted_placeholders(text)
    merged = merge_predictions(ner_only, reg)

    gender_report[pdf_path.name] = report_gender(text)

    tp_n, fp_n, fn_n = span_prf_counts(gold, ner_only)
    tp_m, fp_m, fn_m = span_prf_counts(gold, merged)
    for i, v in enumerate((tp_n, fp_n, fn_n)):
        totals_ner[i] += v
    for i, v in enumerate((tp_m, fp_m, fn_m)):
        totals_merged[i] += v

    p_n, r_n, f_n = prf(tp_n, fp_n, fn_n)
    p_m, r_m, f_m = prf(tp_m, fp_m, fn_m)
    print(f"{pdf_path.name}:")
    print(f"  NER only:   P={p_n:.3f} R={r_n:.3f} F1={f_n:.3f}")
    print(f"  NER+regex:  P={p_m:.3f} R={r_m:.3f} F1={f_m:.3f}")
    gi = gender_report[pdf_path.name]
    conflict_note = ", CONFLICT" if gi.get("conflict") else ""
    print(f"  GENDER (derived, not scored): {gi['value']} via {gi['source']} "
          f"(confidence={gi['confidence']:.2f}{conflict_note})")

    gold_by_label, pred_by_label = {}, {}
    for e in gold:
        gold_by_label.setdefault(e["label"], set()).add((e["start"], e["end"]))
    for e in merged:
        pred_by_label.setdefault(e["label"], set()).add((e["start"], e["end"]))
    for label in set(gold_by_label) | set(pred_by_label):
        g, p = gold_by_label.get(label, set()), pred_by_label.get(label, set())
        tp, fp, fn = len(g & p), len(p - g), len(g - p)
        d = per_label_merged.setdefault(label, [0, 0, 0])
        d[0] += tp; d[1] += fp; d[2] += fn

p_n, r_n, f_n = prf(*totals_ner)
p_m, r_m, f_m = prf(*totals_merged)
print("\n=== OVERALL across all 5 org_data PDFs ===")
print(f"NER only:   Precision={p_n:.3f}  Recall={r_n:.3f}  F1={f_n:.3f}")
print(f"NER+regex:  Precision={p_m:.3f}  Recall={r_m:.3f}  F1={f_m:.3f}")

print("\n=== Per-label breakdown (NER+regex), exact span match ===")
for label, (tp, fp, fn) in sorted(per_label_merged.items()):
    p, r, f = prf(tp, fp, fn)
    print(f"  {label:16s} P={p:.3f}  R={r:.3f}  F1={f:.3f}  (tp={tp} fp={fp} fn={fn})")

print("\n=== GENDER (derived from INSZ parity, informational only -- not a trained/scored label) ===")
for fname, gi in gender_report.items():
    print(f"  {fname:24s} {gi['value']} via {gi['source']} (confidence={gi['confidence']:.2f})")